In [1]:
import toml
from pathlib import Path
config_path = Path("../../../../configuration/asim_configuration")
input_config = toml.load(config_path/ "input_configuration.toml")
summary_config = toml.load(config_path/ "summary_configuration.toml")

compare between model results and survey data: see how the results change/improve when we use coefficients estimated from our survey
- data: persons
- variables: distance_to_work, distance_to_school, is_worker, is_grade/highschool/university
- distance bins from [workplace_location.csv](https://github.com/psrc/psrc_activitysim/blob/main/configs_dev/workplace_location.csv)



In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [3]:
# read data
person = util.get_person_data(summary_config, uncloned=True)
hh_data = util.get_hh_data(summary_config, uncloned=True)

per_data = person.merge(hh_data,how="left",on=['household_id','source'])
# remove workers that have a distance of -1
per_data = per_data[(per_data['distance_to_work'] > -1) & (~per_data['work_from_home'])]

## Distance to work

In [5]:
def plot_distance(df:pd.DataFrame, group:str, title_name:str):

    count = df.loc[(df['source']=='model results') & (df[group]),['distance_to_work_bin']].value_counts()
    print(f"model person count =\n"
          f"{count.sort_values()}")

    # plot1
    df_plot = df.loc[(df[group]) & (df['distance_to_work_bin'] != float('nan'))].groupby(['source','distance_to_work_bin'])['person_weight'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['person_weight'].\
            apply(lambda x: 100 * x / float(x.sum()))
    # df_plot
    #
    fig1 = px.bar(df_plot, x='distance_to_work_bin', y="percentage", color="source", barmode="group",
                  title=title_name)
    fig1.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
    fig1.update_layout(height=400, width=700, font=dict(size=11))
    fig1.show()

    # plot2
    df_plot = per_data.loc[(per_data[group]) & (per_data['distance_to_work_bin_60mi'].notna())].groupby(['source','distance_to_work_bin_60mi'])['person_weight'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['person_weight'].\
                apply(lambda x: 100 * x / float(x.sum()))
    # df_plot
    #
    fig2 = px.line(df_plot, x='distance_to_work_bin_60mi', y="percentage", color="source",
                   title=title_name)
    fig2.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
    fig2.update_layout(height=400, width=700, font=dict(size=11))
    fig2.show()

# distance to work
plot_distance(per_data,'is_worker',"worker: distance to work")

model person count =
Series([], Name: count, dtype: int64)


C:\Users\Modeller\AppData\Local\Temp\ipykernel_8124\4048670242.py:20: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



## Distance to work by segments

In [6]:
df_plot = per_data.loc[(per_data['is_worker']) & (per_data['distance_to_work_bin'] != float('nan'))].\
    groupby(['source','income_group','distance_to_work_bin'])['person_weight'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source','income_group'], group_keys=False)['person_weight']. \
    apply(lambda x: 100 * x / float(x.sum()))
# df_plot

fig = px.bar(df_plot, x="distance_to_work_bin", y="percentage", color="source",barmode="group",
             facet_col="income_group", facet_col_wrap=2,
             title="Distance to work by income group")
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
fig.update_layout(height=600, width=700, font=dict(size=11))
fig.show()

C:\Users\Modeller\AppData\Local\Temp\ipykernel_8124\2731469913.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Modeller\AppData\Local\Temp\ipykernel_8124\2731469913.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [7]:
df_plot = per_data.loc[(per_data['is_worker']) & (per_data['distance_to_work_bin_60mi'] != float('nan'))].groupby(['source','income_group','distance_to_work_bin_60mi'])['person_weight'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source','income_group'], group_keys=False)['person_weight']. \
    apply(lambda x: 100 * x / float(x.sum()))
# df_plot

fig2 = px.line(df_plot, x='distance_to_work_bin_60mi', y="percentage", color="source",
               facet_col="income_group", facet_col_wrap=2,
               title="Distance to work by income group")
fig2.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
fig2.update_layout(height=400, width=700, font=dict(size=11))
fig2.show()

C:\Users\Modeller\AppData\Local\Temp\ipykernel_8124\944872725.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Modeller\AppData\Local\Temp\ipykernel_8124\944872725.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



- Household density represents the household density of a person's home location

In [8]:
df_plot = per_data.loc[(per_data['is_worker']) & (per_data['distance_to_work_bin'] != float('nan'))].groupby(['source','log_hh_1_group','distance_to_work_bin'])['person_weight'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source','log_hh_1_group'], group_keys=False)['person_weight']. \
    apply(lambda x: 100 * x / float(x.sum()))
# df_plot

fig = px.bar(df_plot, x="distance_to_work_bin", y="percentage", color="source",barmode="group",
             facet_col="log_hh_1_group", facet_col_wrap=2,
             title="Distance to work by household density")
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
fig.update_layout(height=600, width=700, font=dict(size=11))
fig.show()

C:\Users\Modeller\AppData\Local\Temp\ipykernel_8124\664459451.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Modeller\AppData\Local\Temp\ipykernel_8124\664459451.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [9]:
df_plot = per_data.loc[(per_data['is_worker']) & (per_data['distance_to_work_bin_60mi'] != float('nan'))].groupby(['source','log_hh_1_group','distance_to_work_bin_60mi'])['person_weight'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source','log_hh_1_group'], group_keys=False)['person_weight']. \
    apply(lambda x: 100 * x / float(x.sum()))
# df_plot

fig2 = px.line(df_plot, x='distance_to_work_bin_60mi', y="percentage", color="source",
               facet_col="log_hh_1_group", facet_col_wrap=2,
               title="Distance to work by household density")
fig2.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
fig2.update_layout(height=400, width=700, font=dict(size=11))
fig2.show()

C:\Users\Modeller\AppData\Local\Temp\ipykernel_8124\1849919757.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Modeller\AppData\Local\Temp\ipykernel_8124\1849919757.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

